In [ ]:
# !pip install -U langchain-community
# !pip install -qU langchain-openai
# !pip install --upgrade langchain-community duckduckgo-search
# !pip install langgraph
# !pip install requests pubchempy rdkit-pypi
# !pip install ctx-python
# !pip install biopython
# !pip install langgraph-checkpoint-sqlite
# !pip install faiss-cpu
# !pip install langchain_openai python-dotenv
# !pip install "numpy<2"  # Use NumPy 1.x for compatibility with rdkit-pypi
# !pip install beautifulsoup4
# !pip install elsapy
# !pip install --upgrade numpy
# !pip install docling
# !pip install langchain-core==1.0.0
# !pip install --upgrade langchain
# !pip install sentence-transformers

# !pip install pdfplumber
# !pip install PyMuPDF
# !pip install --upgrade rdkit
# !pip install --upgrade --force-reinstall numpy spacy thinc flair
# !pip install docling docling-parse docling-core rapidocr-onnxruntime onnxruntime biopython requests pydantic bs4
# !pip install httpx[http2]
# !pip install pyab3p
# !pip install springernature-api-client
# !pip install lancedb
# !pip install pypdf
# !pip install pdfminer.six

In [ ]:
import sys
import os
import pandas as pd
import uuid
import shutil
from pathlib import Path
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage

load_dotenv()

# Robustly add code directory to path
current_dir = os.getcwd()
# Check if we are in notebooks directory or root
if os.path.basename(current_dir) == 'notebooks':
    code_dir = os.path.abspath(os.path.join(current_dir, '../code'))
else:
    # Assuming we are in root or somewhere else, try to find code relative to the notebook file location if possible
    # But since we can't easily get notebook path in all kernels, we'll try common locations
    if os.path.exists(os.path.join(current_dir, 'code')):
        code_dir = os.path.abspath(os.path.join(current_dir, 'code'))
    elif os.path.exists(os.path.join(current_dir, '../code')):
        code_dir = os.path.abspath(os.path.join(current_dir, '../code'))
    else:
        # Fallback to hardcoded path if needed, or just assume ../code relative to where we think we are
        code_dir = os.path.abspath('../code')

if code_dir not in sys.path:
    sys.path.append(code_dir)

print(f"Added {code_dir} to sys.path")

try:
    from agent_graph import build_agent, determine_final_claim
    print("Successfully imported agent_graph")
except ImportError as e:
    print(f"Error importing agent_graph: {e}")
    print(f"Current sys.path: {sys.path}")

# Setup logging
import logging
logging.basicConfig(level=logging.WARNING)
log = logging.getLogger("dbp_eval")

Added /home/chen/proj/db_form_agent_system/code to sys.path


/home/chen/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2025-12-17 04:50:44,269 | INFO | Enabling RDKit 2022.09.5 jupyter extensions


Successfully imported agent_graph


## Load Data

In [ ]:
file_path = "../es4c13900_si_001.xlsx"
df = pd.read_excel(file_path, sheet_name='TableS1_716 Confirmed DBPs', header=1)

# Clean up column names if needed
df.columns = [c.strip() for c in df.columns]

print(f"Total rows: {len(df)}")
df[['IUPAC name', 'Preferred DBP name']].head()

Total rows: 716


,IUPAC name,Preferred DBP name
0,"2,2,2-trichloroacetaldehyde",Chloral (Trichloroethanal; Trichloroacetaldehy...
1,"2,6-Dibromo-4-[1-[3,5-dibromo-4-(2,6-dibromo-4...","2,6-Dibromo-4-(2-(3,5-Dibromo-4-(2,6-Dibromo-4..."
2,"(1S,2R,3R,4S,5R,8R,9S,10S,11R,12R,14R)-11-[(2S...",Anhydroerythromycin (E-DBP1)
3,NaN,Bromonaproxen
4,"L-Alanine, N-(3,5-diiodo-L-tyrosyl)- (9CI)","3,5-Diiodo-Tyrosylalanine (3,5-Di-I-Tyr-Ala)"


## Define Evaluation Function

In [ ]:
def evaluate_dbp(compound_name: str, base_results_dir: str = "../searching_results") -> dict:
    """
    Runs the agent to check if a compound is a Disinfection By-Product.
    """
    session_id = str(uuid.uuid4())
    session_dir = os.path.join(base_results_dir, f"dbp_eval_{session_id}")
    os.makedirs(session_dir, exist_ok=True)
    
    print(f"Evaluating: {compound_name}")
    
    # Build the agent
    app = build_agent(base_results_dir)
    
    # Construct Query
    query = f"According to the following definition: A disinfection byproduct (DBP) is a chemical compound that is formed incidentally by the reaction of a disinfectant with organic and inorganic matter present in water, wastewater, or other matrices. To be considered 'experimentally confirmed', there must be clear evidence from a peer-reviewed study demonstrating its formation during a disinfection process and its chemical identification. Is the compound '{compound_name}' a confirmed DBP? Answer: 'Confirmed DBP' or 'Not Confirmed DBP'"
    
    initial_state = {
        "messages": [HumanMessage(content=query)],
        "original_query": query,
        "search_session_dir": session_dir,
        "stop_history": [],
        "consecutive_no_new_evidence": 0,
        "consecutive_tool_unavailable": 0,
        "last_researcher_json": None,
        "last_critique_json": None,
        "claim_history": [],
        "confidence_history": []
    }
    
    config = {"configurable": {"thread_id": session_id, "search_session_dir": session_dir}}
    
    # Run Agent
    try:
        final_state = app.invoke(initial_state, config=config)
        final_claim = determine_final_claim(final_state["messages"])
    except Exception as e:
        print(f"Error running agent: {e}")
        final_claim = "Error"
    
    # Normalize Claim
    predicted = "Unclear"
    claim_lower = final_claim.lower()
    
    # Check for negative first to avoid matching "confirmed" in "not confirmed"
    if "not confirmed" in claim_lower:
        predicted = "No"
    elif "confirmed" in claim_lower:
        predicted = "Yes"
            
    print(f"  -> Prediction: {final_claim} (Normalized: {predicted})")
    
    return {
        "compound": compound_name,
        "agent_claim": final_claim,
        "predicted_normalized": predicted,
        "session_dir": session_dir
    }

## Run on Sample

Running on a small sample of 5 compounds to verify the workflow.

In [ ]:
 Filter out NaNs in IUPAC name
df_clean = df.dropna(subset=['IUPAC name'])

# Sample 5 compounds
sample_df = df_clean.sample(5, random_state=42)

results = []

for idx, row in sample_df.iterrows():
    compound = row['IUPAC name']
    res = evaluate_dbp(compound)
    results.append(res)

results_df = pd.DataFrame(results)
results_df

## Save Results

In [ ]:
results_df.to_csv("dbp_evaluation_sample_results.csv", index=False)
print("Results saved to dbp_evaluation_sample_results.csv")